In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
import json
from tqdm import tqdm
import re

model_name = "gpt2-large"
output_dir = "./gpt2-large-rm"
tokenized_cache_dir = "./rm_tokenized_ultra_v1"

checkpoint_dir = "/content/drive/MyDrive/rm-checkpoints/gpt2-large-rm"
os.makedirs(checkpoint_dir, exist_ok=True)

save_every_steps = 5000
max_length = 512
batch_size = 1
num_epochs = 1
learning_rate = 2e-6
warmup_ratio = 0.05
logit_l2_weight = 0.01

data_path = "/content/drive/MyDrive/ultrafeedback_1024_filtered.jsonl"

print("\n=== Config ===")
print("model:", model_name)
print("dataset:", data_path)

device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

def get_latest_checkpoint(dir_path):
    pattern = re.compile(r"checkpoint-step-(\d+)")
    best = None
    max_step = -1

    if not os.path.exists(dir_path):
        return None, None

    for name in os.listdir(dir_path):
        m = pattern.match(name)
        if m:
            step = int(m.group(1))
            if step > max_step:
                max_step = step
                best = os.path.join(dir_path, name)

    return best, max_step

raw_tokenizer = AutoTokenizer.from_pretrained(model_name)

special_tokens = {
    "additional_special_tokens": ["<|prompt|>", "<|assistant|>"]
}
raw_tokenizer.add_special_tokens(special_tokens)

if raw_tokenizer.pad_token is None:
    raw_tokenizer.pad_token = raw_tokenizer.eos_token


resume_ckpt, resume_step = get_latest_checkpoint(checkpoint_dir)

if resume_ckpt:
    print("🔄 Resuming from:", resume_ckpt)
    tokenizer = AutoTokenizer.from_pretrained(resume_ckpt)
    model = AutoModelForSequenceClassification.from_pretrained(
        resume_ckpt, num_labels=1)
    step = resume_step
else:
    print("🆕 Training from scratch")
    tokenizer = raw_tokenizer
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=1)
    step = 0

model.resize_token_embeddings(len(tokenizer))
model.to(device)

def extract_assistant(messages):
    """从 messages list 里取最后一个 assistant 的 content"""
    return messages[-1]["content"]


class UltraFeedbackDataset(Dataset):
    def __init__(self, path):
        self.data = []
        with open(path) as f:
            for line in f:
                row = json.loads(line)

                prompt = row["prompt"]
                chosen = extract_assistant(row["chosen"])
                rejected = extract_assistant(row["rejected"])

                self.data.append({
                    "prompt": prompt,
                    "chosen": chosen,
                    "rejected": rejected
                })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]

        full_chosen = f"<|prompt|>\n{row['prompt']}\n\n<|assistant|>\n{row['chosen']}"
        full_rejected = f"<|prompt|>\n{row['prompt']}\n\n<|assistant|>\n{row['rejected']}"

        chosen_tok = tokenizer(
            full_chosen,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt"
        )
        rejected_tok = tokenizer(
            full_rejected,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt"
        )

        return {
            "chosen_input_ids": chosen_tok["input_ids"][0],
            "chosen_attention_mask": chosen_tok["attention_mask"][0],
            "rejected_input_ids": rejected_tok["input_ids"][0],
            "rejected_attention_mask": rejected_tok["attention_mask"][0],
        }

train_data = UltraFeedbackDataset(data_path)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)

print("Dataset size:", len(train_data))

def rm_loss(r_c, r_r, l2_weight=0.01):
    base = torch.nn.functional.softplus(-(r_c - r_r)).mean()
    l2 = l2_weight * (r_c.pow(2).mean() + r_r.pow(2).mean())
    return base + l2

optimizer = AdamW(model.parameters(), lr=learning_rate)

total_steps = len(train_loader) * num_epochs
warmup_steps = int(warmup_ratio * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer, warmup_steps, total_steps
)

print("Total steps:", total_steps)

model.train()

print("\n🔥 Start RM Training ...\n")

for epoch in range(num_epochs):
    for batch in train_loader:

        optimizer.zero_grad()

        r_c = model(
            input_ids=batch["chosen_input_ids"].to(device),
            attention_mask=batch["chosen_attention_mask"].to(device)
        ).logits.squeeze(-1)

        r_r = model(
            input_ids=batch["rejected_input_ids"].to(device),
            attention_mask=batch["rejected_attention_mask"].to(device)
        ).logits.squeeze(-1)

        loss = rm_loss(r_c, r_r, logit_l2_weight)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)

        optimizer.step()
        scheduler.step()

        if step % 50 == 0:
            pref_acc = (r_c > r_r).float().mean().item()
            print(f"Step {step:6d} | Loss={loss.item():.4f} | Pref-Acc={pref_acc:.3f}")

        if step % save_every_steps == 0 and step > 0:
            ckpt = os.path.join(checkpoint_dir, f"checkpoint-step-{step}")
            model.save_pretrained(ckpt)
            tokenizer.save_pretrained(ckpt)

        step += 1

print("\n🎉 Training Complete")

os.makedirs(output_dir, exist_ok=True)
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print("Saved final RM to:", output_dir)


In [ ]:
!pip install trl accelerate transformers datasets --upgrade

In [ ]:
import os
import torch
import torch.nn as nn
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
)
from trl import PPOTrainer, PPOConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

ppo_config = PPOConfig(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,

    # PPO-specific
    num_ppo_epochs=4,
    mini_batch_size=1,
    kl_coef=0.1,
    cliprange=0.2,
    vf_coef=0.1,
    gamma=1.0,
    lam=0.95,

    # generation configs
    response_length=60,
    temperature=0.7,
    report_to="none",
)

policy_name = "gpt2-medium"
rm_path = "/content/drive/MyDrive/rm-checkpoints/gpt2-large-rm/checkpoint-step-55000"


tokenizer = AutoTokenizer.from_pretrained(policy_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

policy = AutoModelForCausalLM.from_pretrained(
    policy_name
).to(device)

ref_model = AutoModelForCausalLM.from_pretrained(
    policy_name
).to(device)

policy.resize_token_embeddings(len(tokenizer))
ref_model.resize_token_embeddings(len(tokenizer))

policy.config.pad_token_id = tokenizer.pad_token_id
ref_model.config.pad_token_id = tokenizer.pad_token_id

reward_model = AutoModelForSequenceClassification.from_pretrained(
    rm_path,
    num_labels=1
).to(device)

value_model = AutoModelForSequenceClassification.from_pretrained(
    policy_name, num_labels=1
)

uf_path = "/content/drive/MyDrive/ultrafeedback_1024_filtered.jsonl"

# jsonl -> HF Dataset
raw_dataset = load_dataset(
    "json",
    data_files=uf_path,
    split="train[:1%]"
)

print("Raw UF dataset size:", len(raw_dataset))
print("Example row:", raw_dataset[0])

def build_prompt(sample):
    return {"prompt": sample["prompt"]}

dataset = raw_dataset.map(
    build_prompt,
    remove_columns=[col for col in raw_dataset.column_names if col != "prompt"]
)

print("After build_prompt columns:", dataset.column_names)
print("Example prompt:", dataset[0]["prompt"][:200])
def tokenize_fn(examples):
    return tokenizer(
        examples["prompt"],
        truncation=True,
        padding="max_length",
        max_length=256,
        return_attention_mask=True,
    )

dataset = dataset.map(tokenize_fn, batched=True)


data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True)
dataset = dataset.remove_columns(
    [col for col in dataset.column_names if col not in ["input_ids", "attention_mask"]]
)
dataset = dataset.with_format(type="torch")

print("Dataset type:", type(dataset))
print("Example item:", dataset[0])
print("Length:", len(dataset))

trainer = PPOTrainer(
    args=ppo_config,
    processing_class=tokenizer,
    model=policy,
    ref_model=ref_model,
    reward_model=reward_model,
    value_model=value_model,
    train_dataset=dataset,
    eval_dataset=dataset,
    data_collator=data_collator
)


print("🔥 Starting PPO training...")
trainer.train()
print("🎉 PPO Finished!")

trainer.save_model(ppo_config.output_dir or "tiny-ppo-output")
print("Saved final model.")


In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

rm_path = "/content/drive/MyDrive/rm-checkpoints/gpt2-large-rm/checkpoint-step-80000"

reward_model = AutoModelForSequenceClassification.from_pretrained(
    rm_path,
    torch_dtype=torch.float16,
    local_files_only=True,
).cuda()

tokenizer = AutoTokenizer.from_pretrained(rm_path)

text = "<|prompt|>\nHello, how are you?\n\n<|assistant|>\nI am fine."

enc = tokenizer(text, return_tensors="pt").to("cuda")

with torch.no_grad():
    out = reward_model(**enc)

print("output:", out)
print("logits:", out.logits)
print("logits shape:", out.logits.shape)
